---
**Hypothesis Testing & t-Tests in Python**
Data Analysis Course · Week 7
---

This notebook is the Python equivalent of the R Markdown `_06_hypothesis_testing.Rmd`.
Topics: formulating **H0/H1 hypotheses**, one- and two-sample **t-tests**, one- vs. two-sided tests,
checking **normality** (Shapiro-Wilk), and building the H0 distribution empirically.

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `numpy`, `pandas`, `matplotlib`, `scipy`
```
pip install numpy pandas matplotlib scipy
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## 0 – Recap of the previous sheets

In the last two sheets we learned about probability distributions, the central limit theorem and
confidence intervals.

## 1 – Introduction and objectives

Here we start to *formulate hypotheses* and *test* them — mainly focusing on tests of **mean values**.

## 2 – First statistical test

We work again with the diabetes dataset.

In [ ]:
dat = pd.read_csv("https://tinyurl.com/y4fark9g", sep="\t")
dat = dat.set_index("id")
dat.describe(include="all")

Let's check how weight is distributed between males and females:

In [ ]:
# R: rows.men = which(dat$gender=='male') — in pandas we filter directly
weight_men = dat.loc[dat["gender"] == "male", "weight"].dropna()
weight_women = dat.loc[dat["gender"] == "female", "weight"].dropna()

plt.hist(weight_men, bins=15, density=True, alpha=0.6, color="blue", label="men")
plt.hist(weight_women, bins=15, density=True, alpha=0.6, color="orange", label="women")
plt.xlabel("Weight")
plt.title("Histogram of weight")
plt.legend()
plt.show()

In [ ]:
mean_men = weight_men.mean()
mean_women = weight_women.mean()

plt.hist(weight_men, bins=15, density=True, alpha=0.6, color="blue", label="men")
plt.hist(weight_women, bins=15, density=True, alpha=0.6, color="orange", label="women")
plt.axvline(mean_men, color="blue", linestyle=":", linewidth=3)
plt.axvline(mean_women, color="orange", linestyle=":", linewidth=3)
plt.xlabel("Weight")
plt.title("Histogram of weight")
plt.legend()
plt.show()

# Do you think the mean of the male weight is lower than 180?
# Do you think the female mean differs from the male mean?
# Do you think male weight is higher than female weight, on average?

Now we use a statistical test to check whether:
- males have a mean weight **significantly lower** than a specific value (one-sample, one-sided test)
- there is a **significant** difference between the two groups' means (two-sample, two-sided test)
- males have a **significantly higher** mean weight than females (two-sample, one-sided test)

This is exactly what mean tests such as the **t-test** are designed for.

### One-sample t-test

A one-sample t-test checks if values differ significantly from a target value. Example: sample 10
chocolate bars and test whether they significantly differ from the expected weight of 100 g.

> Should we perform a one- or two-sided test?

In [ ]:
bars = np.array([103, 103, 97, 102.5, 100.5, 103, 101.3, 99.5, 101, 104])
chocbar_mean = 100

`scipy.stats.ttest_1samp()` offers the `alternative` parameter: `'two-sided'`, `'less'` or `'greater'`
(R's `t.test(..., alternative=)` — same three options, just spelled with a hyphen instead of a dot).

It is essential to **clearly formulate H0 and H1**:

> H0: the expectation value of "Weight of a chocolate bar" is **equal** to 100 g.
> H1: the expectation value of "Weight of a chocolate bar" is **different** from 100 g.

In [ ]:
# R: t.test(x = bars, mu = chocbar.mean, alternative = "two.sided")
result = stats.ttest_1samp(bars, popmean=chocbar_mean, alternative="two-sided")
print(result)
print("p-value:", result.pvalue)

# Can you reject H0 at alpha=0.05? At alpha=0.1?

Using alpha = 0.05, H0 cannot be rejected (p ≈ 0.052 ≥ 0.05): the mean is not significantly
different from 100 g. Using alpha = 0.1, H0 **can** be rejected. **But** alpha should never be
chosen *after* seeing the result — decide on H0/H1 and alpha *before* running the test!

Regarding male weight: we want to check if males have a mean weight **significantly lower** than
180 — a **one-sample, one-sided** test with `mu=180`, `alternative='less'`.

> H0: the expectation value of "Weight of male patients" is **equal to or greater than** 180.
> H1: the expectation value of "Weight of male patients" is **less than** 180.

In [ ]:
result = stats.ttest_1samp(weight_men, popmean=180, alternative="less")
print(result)

# How would you interpret this at alpha=0.05? Can you reject H0?

### Two-sample t-test (two-sided)

Now we compare the mean weights of males and females — a **two-sample, two-sided** t-test.

> H0: the expectation value of "Weight of male patients" is **equal** to that of "Weight of female patients".
> H1: they are **different**.

In [ ]:
# R: t.test(weight.men, weight.women)  — alternative="two-sided" is scipy's default too
result = stats.ttest_ind(weight_men, weight_women, equal_var=False)   # Welch's t-test, like R's default
print(result)

# How would you interpret this at alpha=0.05? Can you reject H0?

### Two-sample t-test (one-sided)

> H0: the expectation value of male weight is **equal to or lower than** female weight.
> H1: male weight is **higher** than female weight.

In [ ]:
result = stats.ttest_ind(weight_men, weight_women, equal_var=False, alternative="greater")
print(result)

# How would you interpret this at alpha=0.05?
# Can you explain why the one-sided p-value is lower than the two-sided p-value?

In [ ]:
two_sided = stats.ttest_ind(weight_men, weight_women, equal_var=False, alternative="two-sided").pvalue
one_sided = stats.ttest_ind(weight_men, weight_women, equal_var=False, alternative="greater").pvalue
print(two_sided, one_sided)
print(two_sided / one_sided)   # exactly 2, when the observed difference points the right way

This can be visualized with the t-distribution. Say `t = 1.8453` and `df = 372.45` (see the result above).

In [ ]:
# No need to understand every line — just look at the graph and shaded area.
t_obs_demo, df_demo = 1.8453, 372.45
x = np.arange(-5, 5.01, 0.01)
y = stats.t.pdf(x, df=df_demo)

fig, ax = plt.subplots()
ax.plot(x, y, linewidth=3)
ax.axvline(t_obs_demo, color="blue", linestyle=":", linewidth=2)
mask = x >= t_obs_demo
ax.fill_between(x[mask], y[mask], color="purple", alpha=0.8)
plt.title("One-tailed p-value: area for t > 1.8453")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, y, linewidth=3)
ax.axvline(t_obs_demo, color="blue", linestyle=":", linewidth=2)
ax.axvline(-t_obs_demo, color="blue", linestyle=":", linewidth=2)
mask_hi = x >= t_obs_demo
mask_lo = x <= -t_obs_demo
ax.fill_between(x[mask_hi], y[mask_hi], color="purple", alpha=0.8)
ax.fill_between(x[mask_lo], y[mask_lo], color="purple", alpha=0.8)
plt.title("Two-tailed p-value: area for |t| > 1.8453 — exactly double the one-tailed area")
plt.show()

# IMPORTANT: a t-test requires the data to be approximately normally distributed!
# We haven't checked that yet — see "Going further" below.

---
### Exercise 1

Consider the height data below. Formulate H0 and H1, and perform a (one-sided) t-test. Interpret
with alpha=0.05.

In [ ]:
height_men = dat.loc[dat["gender"] == "male", "height"].dropna()
height_women = dat.loc[dat["gender"] == "female", "height"].dropna()

plt.hist(height_men, bins=20, density=True, alpha=0.6, color="blue", label="men")
plt.hist(height_women, bins=20, density=True, alpha=0.6, color="orange", label="women")
plt.axvline(height_men.mean(), color="blue", linestyle=":", linewidth=3)
plt.axvline(height_women.mean(), color="orange", linestyle=":", linewidth=3)
plt.legend()
plt.show()

# Your code here:

### Exercise 2

1. Calculate the mean age of the men.
2. Compare it to age = 50. Formulate H0/H1 and perform a one-sided t-test. Interpret with alpha=0.05.

In [ ]:
# Your code here:

## 3 – Going further: checking normality

t-tests require approximately normally distributed data. If not, we use **non-parametric** tests
(next week). The **Shapiro-Wilk** test checks normality — `scipy.stats.shapiro()` is R's
`shapiro.test()`.

In [ ]:
# QQ-plot reminder
stats.probplot(dat["weight"].dropna(), dist="norm", plot=plt)
plt.title("weight")
plt.show()

# Shapiro-Wilk test
sw = stats.shapiro(dat["weight"].dropna())
print(sw)
print("p-value:", sw.pvalue)
# Reminder: p >= 0.05 -> normally distributed. p < 0.05 -> NOT normally distributed.

In [ ]:
# Apply Shapiro-Wilk to every numeric column
numeric_cols = ["chol", "stab.glu", "hdl", "age", "height", "weight", "bp.1s", "bp.1d",
                 "waist", "hip", "glyhb", "ratio"]
numeric_cols = [c for c in numeric_cols if c in dat.columns]

for col in numeric_cols:
    p = stats.shapiro(dat[col].dropna()).pvalue
    print(f"{col}: p = {p:.2e}")

# Apparently none of the parameters is perfectly normal.
# For the training, we'll continue with "bp.1d" which doesn't look too bad.

In [ ]:
stats.probplot(dat["bp.1d"].dropna(), dist="norm", plot=plt)
plt.title("Blood pressure")
plt.show()

We test bp.1d between men and women:

> H0: the bp.1d mean of men is not significantly different from that of women.
> H1: the bp.1d mean of men is significantly different from that of women.

In [ ]:
bp_men = dat.loc[dat["gender"] == "male", "bp.1d"].dropna()
bp_women = dat.loc[dat["gender"] == "female", "bp.1d"].dropna()

plt.boxplot([bp_men, bp_women], tick_labels=["men", "women"])
plt.title("bp.1d values")
plt.show()

In [ ]:
result = stats.ttest_ind(bp_men, bp_women, equal_var=False, alternative="two-sided")
print(result)

t_obs = result.statistic
print("t.obs:", t_obs)

# With alpha=0.05, H0 cannot be rejected!

## 4 – Extra insight: building the H0 distribution

The t-test relies on a *theoretical* distribution under H0. We can build this H0 distribution
*empirically* using resampling. Rationale: between two **randomly** chosen groups, there's no
reason to expect a systematic difference.

In [ ]:
rng = np.random.default_rng(0)
n_men = len(bp_men)
bp_all = dat["bp.1d"].dropna().values

i_random_men = rng.choice(len(bp_all), size=n_men, replace=False)
mask = np.zeros(len(bp_all), dtype=bool)
mask[i_random_men] = True

t_h0 = stats.ttest_ind(bp_all[mask], bp_all[~mask]).statistic
print(t_h0)

# Repeat this several times to get various t-values across random groups.

In [ ]:
t_values = []
for _ in range(10000):
    i_random_men = rng.choice(len(bp_all), size=n_men, replace=False)
    mask = np.zeros(len(bp_all), dtype=bool)
    mask[i_random_men] = True
    t_values.append(stats.ttest_ind(bp_all[mask], bp_all[~mask]).statistic)
t_values = np.array(t_values)

plt.hist(t_values, bins=40)
plt.show()

# This distribution approximates the H0 distribution of the test statistic.

In [ ]:
print(t_values.mean())
print(t_values.std(ddof=1))
stats.probplot(t_values, dist="norm", plot=plt)
plt.show()

# This looks like a standard normal / t-distribution with df = n - 2. Do you remember why -2?

In [ ]:
x = np.arange(-5, 5.1, 0.1)
y = stats.t.pdf(x, df=len(bp_all) - 2)

plt.hist(t_values, bins=40, density=True)
plt.plot(x, y, color="red", linewidth=3)
plt.show()

# Pretty close, no?

### Computing the p-value

In [ ]:
plt.hist(t_values, bins=40, range=(-5, 5))
plt.axvline(t_obs, color="red", linestyle=":", linewidth=4)
plt.axvline(-t_obs, color="darkred", linestyle=":", linewidth=1)
plt.xlim(-5, 5)
plt.show()

In [ ]:
# empirical p-value: fraction of random splits at least as extreme as t.obs
p_value = np.mean((t_values < -abs(t_obs)) | (t_values > abs(t_obs)))
p_value

In [ ]:
df = len(bp_all) - 2

## one-sided p-value for the upper tail (P(t > t_obs))
print(stats.t.sf(t_obs, df=df))   # .sf() = "survival function" = 1 - cdf, same as R's lower.tail=FALSE

## two-sided p-value
print(2 * stats.t.sf(t_obs, df=df))

In [ ]:
result = stats.ttest_ind(bp_men, bp_women)
print(result)
print(result.pvalue)

# Pretty close to the empirical estimate above, right?

In [ ]:
# For large df, the t-distribution converges to the standard normal N(0,1)
x = np.arange(-4, 4.1, 0.1)
y = stats.t.pdf(x, df=len(bp_all) - 2)
z = stats.norm.pdf(x)

plt.plot(x, y, color="blue", linewidth=3, label="t")
plt.plot(x, z, color="red", linestyle="--", linewidth=2, label="norm")
plt.title("t vs. norm for large df")
plt.legend()
plt.show()

In [ ]:
# p-value based on the t-distribution
print(2 * stats.t.sf(t_obs, df=df))
# p-value based on the standard normal distribution
print(2 * stats.norm.sf(t_obs))

## Summary: What have we learned?

| R | Python | Purpose |
|---|--------|---------|
| `t.test(x, mu=..., alternative=)` | `stats.ttest_1samp(x, popmean=..., alternative=)` | One-sample t-test |
| `t.test(x, y, alternative=)` | `stats.ttest_ind(x, y, equal_var=False, alternative=)` | Two-sample t-test (Welch by default in R) |
| `alternative = "two.sided"/"less"/"greater"` | `alternative="two-sided"/"less"/"greater"` | Test direction |
| `shapiro.test(x)` | `scipy.stats.shapiro(x)` | Shapiro-Wilk normality test |
| `sapply(df, shapiro.test)` | loop / dict comprehension over columns | Apply per column |
| `dt(x, df)` | `stats.t.pdf(x, df)` | t-distribution density |
| `pt(q, df, lower.tail=FALSE)` | `stats.t.sf(q, df)` | Upper-tail t probability |
| `sample(1:n, k)` | `rng.choice(n, size=k, replace=False)` | Random sampling without replacement |
| `$statistic` / `$p.value` | `.statistic` / `.pvalue` | Extract test result fields |